# CASE 2 — FEMNIST, natural non-IID, JANUS-FL

Same round protocol as CASE 1, over the writer-disjoint FEMNIST split
(fl/splits.py's writer_disjoint_split), reporting BOTH seen-writer and
unseen-writer accuracy (ROADMAP.md §3.4 -- unseen-writer is the
headline metric). **Run `00_Setup_and_Crypto_Tests.ipynb` and
`CASE1_IID_MNIST.ipynb` first** (CASE1 exercises the full round-loop
wiring end-to-end on the simpler dataset before repeating it here).


In [ ]:
import sys, yaml
sys.path.insert(0, '/content/FL-ZKP')
%cd /content/FL-ZKP

with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)


## Load FEMNIST and compute the writer-disjoint split

In [ ]:
from fl.data_noniid import prepare_case2

femnist_cfg = CFG['datasets']['femnist']
case2_data = prepare_case2(seed=CFG['seed'], cache_dir=femnist_cfg['cache_dir'])
print("clients:", len(case2_data['client_train']))
print("server_root writers:", len(case2_data['split'].server_root_writers))
print("unseen_test writers (headline metric source):", len(case2_data['split'].unseen_test_writers))


## Models: num_classes=62 (digits + upper + lower case)

In [ ]:
from fl.models import build_model, flatten_state_dict

models = {name: build_model(name, num_classes=62, in_channels=1) for name in CFG['mvp_scope']['models']}
for name, m in models.items():
    flat, _ = flatten_state_dict(m)
    print(name, "param count:", flat.numel())


## Round loop

Identical structure to `CASE1_IID_MNIST.ipynb`'s round loop -- reuse
that notebook's completed wiring (once finished there) by importing
the same `fl.client` / `fl.server` calls against
`case2_data['client_train'][client_idx]` instead of CASE1's MNIST
client datasets. Not duplicated verbatim here to avoid the two
notebooks silently drifting out of sync; see CASE1's "Known gap"
section for the exact steps to wire in.


## Evaluation: seen-writer vs. unseen-writer accuracy

In [ ]:
import torch

def evaluate(model, dataset, model_shapes=None):
    model.eval()
    with torch.no_grad():
        logits = model(dataset.images)
        preds = logits.argmax(dim=1)
        acc = (preds == dataset.labels).float().mean().item()
    return acc

for name, model in models.items():
    seen_accs = [evaluate(model, d) for d in case2_data['client_val']]
    seen_writer_acc = sum(seen_accs) / len(seen_accs) if seen_accs else float('nan')
    unseen_writer_acc = evaluate(model, case2_data['unseen_test'])
    print(f"{name}: seen-writer acc={seen_writer_acc:.3f} (ROADMAP.md target >=0.78), "
          f"unseen-writer acc={unseen_writer_acc:.3f} (target >=0.70, this is the headline number)")
